# Etapa I - Importación de datos

Descarga el CSV **Ubicación e información sobre contenedores para la recepción de residuos reciclables** (dataset *Contenedores, programas para la recepción de residuos reciclables y condiciones de recepción (¿Dónde Reciclo?)*, catálogo de datos abiertos de Uruguay) y lo guarda **sin modificar** en `docs/Datos/Dataset - original`. La limpieza se hace en la Etapa II.

In [1]:
from pathlib import Path
import requests

URL = (
    "https://catalogodatos.gub.uy/dataset/f7e0ba91-1649-4dd4-a313-e20d45df3f67/"
    "resource/00928700-0b2f-4685-97b3-1dc0d5009d90/download/containers-2022-07-27.csv"
)

# El notebook corre desde su carpeta: docs/Datos/Análisis de datos/Etapa I - Importación de datos
CARPETA_ORIGINAL = Path("..", "..", "Dataset - original").resolve()
ARCHIVO_ORIGINAL = CARPETA_ORIGINAL / URL.rsplit("/", 1)[-1]

In [2]:
CARPETA_ORIGINAL.mkdir(parents=True, exist_ok=True)

if ARCHIVO_ORIGINAL.exists():
    print("Ya existe, no se vuelve a descargar:", ARCHIVO_ORIGINAL)
else:
    respuesta = requests.get(URL, timeout=120)
    respuesta.raise_for_status()
    ARCHIVO_ORIGINAL.write_bytes(respuesta.content)  # bytes tal cual, sin reescribir
    print("Descargado:", ARCHIVO_ORIGINAL, f"({len(respuesta.content):,} bytes)")

Ya existe, no se vuelve a descargar: C:\Users\Usuario\Documents\Proyecto de Datos\Proyecto - Residuos Montevideo DATA URUGUAY\docs\Datos\Dataset - original\containers-2022-07-27.csv


In [3]:
# Control de integridad: el tamaño local debe coincidir con el que informa el catálogo
API_RECURSO = "https://catalogodatos.gub.uy/api/3/action/resource_show?id=00928700-0b2f-4685-97b3-1dc0d5009d90"
tamano_catalogo = int(requests.get(API_RECURSO, timeout=60).json()["result"]["size"])
tamano_local = ARCHIVO_ORIGINAL.stat().st_size
print("Tamaño en el catálogo:", tamano_catalogo, "| local:", tamano_local, "| coincide:", tamano_catalogo == tamano_local)
assert tamano_catalogo == tamano_local

Tamaño en el catálogo: 403960 | local: 403960 | coincide: True


### Huella y procedencia

Se guarda junto al CSV un archivo `.procedencia.json` con la URL, el tamaño y la huella SHA-256 del archivo. La Etapa II compara esa huella antes de leer los datos, para saber que trabaja sobre exactamente este archivo. Si el archivo cambia, esta celda falla.

In [4]:
import hashlib
import json
from datetime import date


def sha256(ruta):
    """Huella SHA-256 de un archivo, leído por bloques."""
    huella = hashlib.sha256()
    with Path(ruta).open("rb") as flujo:
        for bloque in iter(lambda: flujo.read(1024 * 1024), b""):
            huella.update(bloque)
    return huella.hexdigest()


ARCHIVO_PROCEDENCIA = ARCHIVO_ORIGINAL.parent / (ARCHIVO_ORIGINAL.stem + ".procedencia.json")
huella_actual = sha256(ARCHIVO_ORIGINAL)

if ARCHIVO_PROCEDENCIA.exists():
    procedencia = json.loads(ARCHIVO_PROCEDENCIA.read_text(encoding="utf-8"))
    assert procedencia["sha256"] == huella_actual, "El CSV original cambió desde que se registró su huella"
    print("Huella verificada contra la registrada el", procedencia["fecha_registro"])
else:
    procedencia = {
        "archivo": ARCHIVO_ORIGINAL.name,
        "url": URL,
        "bytes": ARCHIVO_ORIGINAL.stat().st_size,
        "bytes_segun_catalogo": tamano_catalogo,
        "sha256": huella_actual,
        "fecha_registro": date.today().isoformat(),
        "nota": "Huella registrada al verificar el archivo descargado. La Etapa II la compara antes de leerlo.",
    }
    ARCHIVO_PROCEDENCIA.write_text(json.dumps(procedencia, ensure_ascii=False, indent=2), encoding="utf-8")
    print("Procedencia registrada en:", ARCHIVO_PROCEDENCIA.name)

print("SHA-256:", huella_actual)

Huella verificada contra la registrada el 2026-10-29
SHA-256: 01ffdfe84b61c6f5a7d579586d1bbac7283e0fafbf8ab5dca30a1f988a9e8dbf


In [5]:
import pandas as pd

df_original = pd.read_csv(ARCHIVO_ORIGINAL, encoding="utf-8-sig")  # el archivo trae BOM
print(df_original.shape)
df_original.head(3)

(2423, 16)


,Id,External,Latitude,Longitude,Site,Address,Location,State,Site type,Public site,Created at,Updated at,Hidden,Latlon,Information,Custom icon active
0,11780,NaN,-34.753762,-55.845820,NaN,"Ruta 87, Manzana 157, Solar 16. Entre Tapes y ...",Salinas,Canelones,NaN,True,2022-07-26 20:21:39 UTC,2022-07-26 20:32:15 UTC,False,NaN,Compostar para utilizar en huerta comunitaria ...,False
1,11779,NaN,-34.592136,-54.124148,Club de La Pedrera (martes previa coordinación...,NaN,La Paloma,Rocha,NaN,True,2022-07-25 19:18:50 UTC,2022-07-26 20:33:53 UTC,False,NaN,Se recicla el residuo ferroso y plástico que c...,False
2,11778,NaN,-34.618292,-54.151271,Placita de Arachania (sábados previa coordinac...,NaN,La Paloma,Rocha,NaN,True,2022-07-25 19:13:49 UTC,2022-07-26 20:34:12 UTC,False,NaN,Se recicla el residuo ferroso y plástico que c...,False
